In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import sklearn
%matplotlib inline

In [2]:
test_df=pd.read_csv('test.csv')

In [3]:
test_df.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format
0,row_00009,PRD-2WLNC8,8.628,Low Fat,0.0167,household,190.72,STORE-HL7,23,Medium,Tier_3,Superstore
1,row_00015,PRD-LV9PLM,6.993,Low Fat,0.0579,Health and Hygiene,261.07,STORE-9RG,28,Small,Tier_2,Standard Supermarket
2,row_00019,PRD-ET6ZI7,NaN,Low Fat,0.1262,FRUITS AND VEGETABLES,112.50,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket
3,row_00020,PRD-6RX35U,14.034,Regular,0.0761,frozen foods,200.71,STORE-DKU,30,NaN,Tier_2,Standard Supermarket
4,row_00023,PRD-I4J38V,13.063,Low Fat,0.0650,Frozen Foods,150.59,STORE-89Z,33,Medium,Tier_1,Standard Supermarket


In [4]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1705 entries, 0 to 1704
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   1705 non-null   str    
 1   product_code         1705 non-null   str    
 2   product_weight_kg    1399 non-null   float64
 3   fat_content          1705 non-null   str    
 4   shelf_visibility     1705 non-null   float64
 5   product_category     1705 non-null   str    
 6   product_price        1705 non-null   float64
 7   store_code           1705 non-null   str    
 8   store_age_years      1705 non-null   int64  
 9   store_size           1214 non-null   str    
 10  store_location_tier  1705 non-null   str    
 11  store_format         1705 non-null   str    
dtypes: float64(3), int64(1), str(8)
memory usage: 283.4 KB


In [5]:
test_df.describe()

,product_weight_kg,shelf_visibility,product_price,store_age_years
count,1399.000000,1705.000000,1705.000000,1705.000000
mean,12.865960,0.068491,143.069490,34.127859
std,4.647713,0.051721,61.888692,8.322650
min,4.693000,0.000000,31.980000,23.000000
25%,8.753000,0.030200,96.560000,28.000000
50%,12.617000,0.056400,144.630000,33.000000
75%,16.915500,0.098400,185.210000,45.000000
max,21.980000,0.302000,270.870000,47.000000


In [6]:
test_df.isnull().sum()

id                       0
product_code             0
product_weight_kg      306
fat_content              0
shelf_visibility         0
product_category         0
product_price            0
store_code               0
store_age_years          0
store_size             491
store_location_tier      0
store_format             0
dtype: int64

In [8]:
test_df['product_weight_kg'] = test_df.groupby('product_code')['product_weight_kg'].transform(lambda x: x.fillna(x.median()))

In [9]:
test_df['product_weight_kg'] = test_df.groupby('product_category')['product_weight_kg'].transform(lambda x: x.fillna(x.median()))

In [10]:
test_df['product_weight_kg'].isna().sum()

np.int64(0)

In [13]:
test_df['store_size_was_missing'] = test_df['store_size'].isna().astype(int)
known = test_df.dropna(subset=['store_size'])
combo_mode = (known.groupby(['store_format', 'store_location_tier'])['store_size'].agg(lambda x: x.mode()[0]))
format_mode = known.groupby('store_format')['store_size'].agg(lambda x: x.mode()[0])
global_mode = known['store_size'].mode()[0]

In [15]:
def fill_store_size(row):
    if pd.notna(row['store_size']):
        return row['store_size']
        
    key = (row['store_format'], row['store_location_tier'])
    
    if key in combo_mode.index:
        return combo_mode[key]
    if row['store_format'] in format_mode.index:
        return format_mode[row['store_format']]
    return global_mode

In [16]:
test_df['store_size'] = test_df.apply(fill_store_size, axis=1)

In [17]:
test_df.isnull().sum()

id                        0
product_code              0
product_weight_kg         0
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size                0
store_location_tier       0
store_format              0
store_size_was_missing    0
dtype: int64

In [20]:
test_df[test_df['store_size']=='Small']

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,store_size_was_missing
1,row_00015,PRD-LV9PLM,6.993,Low Fat,0.0579,Health and Hygiene,261.07,STORE-9RG,28,Small,Tier_2,Standard Supermarket,0
3,row_00020,PRD-6RX35U,14.034,Regular,0.0761,frozen foods,200.71,STORE-DKU,30,Small,Tier_2,Standard Supermarket,1
5,row_00026,PRD-N4RCER,17.392,Regular,0.2659,Seafood,152.00,STORE-T5G,47,Small,Tier_1,Corner Shop,0
7,row_00033,PRD-12310X,8.067,Low Fat,0.0139,Health and Hygiene,185.21,STORE-9RG,28,Small,Tier_2,Standard Supermarket,0
8,row_00034,PRD-3MS556,13.578,Regular,0.0696,BAKING GOODS,149.91,STORE-T5G,47,Small,Tier_1,Corner Shop,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1693,row_08464,PRD-Z8TUBC,20.763,Low Fat,0.0211,Household,74.58,STORE-YLW,35,Small,Tier_1,Standard Supermarket,0
1694,row_08465,PRD-1RXIV4,17.634,Low Fat,0.0491,Snack Foods,170.23,STORE-JOR,34,Small,Tier_3,Corner Shop,1
1699,row_08481,PRD-VQPG4P,10.893,Low Fat,0.0496,others,168.38,STORE-OYG,25,Small,Tier_2,Standard Supermarket,1
1701,row_08489,PRD-0B58ST,16.748,Regular,0.0136,canned,97.26,STORE-JOR,34,Small,Tier_3,Corner Shop,1


In [21]:
test_df.drop('store_size_was_missing', axis=1, inplace=True)

In [22]:
test_df

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format
0,row_00009,PRD-2WLNC8,8.628,Low Fat,0.0167,household,190.72,STORE-HL7,23,Medium,Tier_3,Superstore
1,row_00015,PRD-LV9PLM,6.993,Low Fat,0.0579,Health and Hygiene,261.07,STORE-9RG,28,Small,Tier_2,Standard Supermarket
2,row_00019,PRD-ET6ZI7,12.454,Low Fat,0.1262,FRUITS AND VEGETABLES,112.50,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket
3,row_00020,PRD-6RX35U,14.034,Regular,0.0761,frozen foods,200.71,STORE-DKU,30,Small,Tier_2,Standard Supermarket
4,row_00023,PRD-I4J38V,13.063,Low Fat,0.0650,Frozen Foods,150.59,STORE-89Z,33,Medium,Tier_1,Standard Supermarket
...,...,...,...,...,...,...,...,...,...,...,...,...
1700,row_08483,PRD-58L9K5,17.919,Low Fat,0.0607,Starchy Foods,168.62,STORE-89Z,33,Medium,Tier_1,Standard Supermarket
1701,row_08489,PRD-0B58ST,16.748,Regular,0.0136,canned,97.26,STORE-JOR,34,Small,Tier_3,Corner Shop
1702,row_08491,PRD-RH8H90,9.876,Low Fat,0.0239,health and hygiene,183.35,STORE-HL7,23,Medium,Tier_3,Superstore
1703,row_08504,PRD-J51IFI,11.928,Low Fat,0.0221,Breads,166.09,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket


In [24]:
test_df['product_code'].mode()

0    PRD-02NFGI
Name: product_code, dtype: str

In [26]:
test_df['store_code'].unique()

<ArrowStringArray>
['STORE-HL7', 'STORE-9RG', 'STORE-7WS', 'STORE-DKU', 'STORE-89Z', 'STORE-T5G',
 'STORE-OYG', 'STORE-YLW', 'STORE-AGY', 'STORE-JOR']
Length: 10, dtype: str